In [2]:
import torch

if torch.cuda.is_available():
    print(f"شغال 100%! نوع كرت الشاشة: {torch.cuda.get_device_name(0)}")
else:
    print("مش شغال! الجلسة حالياً على الـ CPU فقط.")

شغال 100%! نوع كرت الشاشة: Tesla T4


In [3]:
!pip install -q mne

import os
import gc
import mne
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, random_split
import numpy as np
from google.colab import drive
from huggingface_hub import hf_hub_download

# 1. ربط Google Drive وتحميل متجهات CLIP الحقيقية
drive.mount('/content/drive')
drive_dir = "/content/drive/MyDrive/Brain_to_Image"
clip_train_embeddings = torch.load(os.path.join(drive_dir, "clip_train_embeddings.pt"))
print(f"تم تحميل متجهات CLIP بنجاح: {clip_train_embeddings.shape}")

# 2. تحميل ومعالجة إشارات الدماغ (sub-01 / ses-01)
print("\nجاري تحميل إشارات الدماغ الخام...")
eeg_path = hf_hub_download(
    repo_id="gasparyanartur/things-eeg2",
    filename="raw-eeg/sub-01/ses-01/raw_eeg_training.npy",
    repo_type="dataset"
)
eeg_dict = np.load(eeg_path, allow_pickle=True).item()

info = mne.create_info(ch_names=eeg_dict['ch_names'], sfreq=eeg_dict['sfreq'], ch_types=eeg_dict['ch_types'])
raw = mne.io.RawArray(eeg_dict['raw_eeg_data'], info, verbose=False)
del eeg_dict
gc.collect()

print("جاري الفلترة (1-40 Hz) وتقليل التردد لـ 250 Hz...")
raw.filter(l_freq=1.0, h_freq=40.0, n_jobs=-1, verbose=False)
raw.resample(250, verbose=False)

events = mne.find_events(raw, stim_channel='stim', verbose=False)
epochs = mne.Epochs(raw, events, tmin=-0.2, tmax=0.8, baseline=(None, 0), preload=True, verbose=False)

eeg_array = epochs.get_data()
labels = epochs.events[:, 2]
del raw, epochs
gc.collect()

# 3. حذف أحداث التنبيه (99999) ومطابقة الإشارات مع الصور الحقيقية
valid_mask = (labels >= 1) & (labels <= 16540)
eeg_array = eeg_array[valid_mask]
img_indices = labels[valid_mask] - 1

# تحويل لـ Tensors وتطبيق Z-Score Standardization لكل قناة
X = torch.from_numpy(eeg_array).float()
X = (X - X.mean(dim=-1, keepdim=True)) / (X.std(dim=-1, keepdim=True) + 1e-6)
Y = clip_train_embeddings[img_indices].float()

del eeg_array, clip_train_embeddings
gc.collect()

print(f"\n=== البيانات جاهزة ومطابقة 100% ===")
print(f"إشارات الدماغ (X): {X.shape}")
print(f"متجهات الصور (Y): {Y.shape}")

# 4. تقسيم البيانات (90% تدريب، 10% تحقق)
dataset = TensorDataset(X, Y)
val_size = int(0.1 * len(dataset))
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
Mounted at /content/drive
تم تحميل متجهات CLIP بنجاح: torch.Size([16540, 768])

جاري تحميل إشارات الدماغ الخام...


raw-eeg/sub-01/ses-01/raw_eeg_training.n(…): reconstructing file:   0%|          |  0.00B / 2.79GB            

raw-eeg/sub-01/ses-01/raw_eeg_training.n(…): downloading bytes:           |  0.00B            

جاري الفلترة (1-40 Hz) وتقليل التردد لـ 250 Hz...

=== البيانات جاهزة ومطابقة 100% ===
إشارات الدماغ (X): torch.Size([16710, 64, 251])
متجهات الصور (Y): torch.Size([16710, 768])


In [4]:
class EEGEncoder(nn.Module):
    def __init__(self, in_channels=64, embed_dim=768):
        super().__init__()
        self.encoder = nn.Sequential(
            # طبقة استخراج الخصائص الزمنية
            nn.Conv1d(in_channels, 128, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.2),

            # طبقة الدمج المكاني والزمني
            nn.Conv1d(128, 256, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.3),

            # طبقة استخراج الأنماط العميقة
            nn.Conv1d(256, 512, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(512),
            nn.GELU(),

            # الحفاظ على 4 نوافذ زمنية رئيسية للإشارة البصرية
            nn.AdaptiveAvgPool1d(4),
            nn.Flatten()
        )
        self.projector = nn.Sequential(
            nn.Linear(512 * 4, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(1024, embed_dim)
        )
        # معامل الحرارة القابل للتعلم لدالة InfoNCE
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

    def forward(self, x):
        features = self.encoder(x)
        embeds = self.projector(features)
        return F.normalize(embeds, dim=-1)

print("تم تعريف معمارية EEGEncoder بنجاح!")

تم تعريف معمارية EEGEncoder بنجاح!


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = EEGEncoder().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)

def clip_contrastive_loss(eeg_embeds, img_embeds, logit_scale):
    scale = torch.clamp(logit_scale.exp(), max=100.0)
    logits = scale * (eeg_embeds @ img_embeds.T)
    targets = torch.arange(len(eeg_embeds), device=eeg_embeds.device)
    loss_e2i = F.cross_entropy(logits, targets)
    loss_i2e = F.cross_entropy(logits.T, targets)
    return (loss_e2i + loss_i2e) / 2.0

print(f"بدء التدريب الفعلي على ({device}) لمدة 40 Epoch...")
best_val_loss = float('inf')

for epoch in range(1, 41):
    # --- مرحلة التدريب ---
    model.train()
    train_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        pred_embeds = model(batch_x)
        loss = clip_contrastive_loss(pred_embeds, batch_y, model.logit_scale)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()
    train_loss /= len(train_loader)

    # --- مرحلة التحقق وحساب الدقة ---
    model.eval()
    val_loss = 0.0
    all_pred, all_true = [], []
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            pred_embeds = model(batch_x)
            loss = clip_contrastive_loss(pred_embeds, batch_y, model.logit_scale)
            val_loss += loss.item()
            all_pred.append(pred_embeds)
            all_true.append(batch_y)
    val_loss /= len(val_loader)

    # حساب Top-1 و Top-5 Retrieval Accuracy على مجموعة التحقق (1654 صورة)
    all_pred = torch.cat(all_pred, dim=0)
    all_true = torch.cat(all_true, dim=0)
    sim_matrix = all_pred @ all_true.T
    ranks = sim_matrix.argsort(dim=-1, descending=True)
    gt = torch.arange(len(all_pred), device=device).unsqueeze(1)
    top1 = (ranks[:, :1] == gt).any(dim=-1).float().mean().item() * 100
    top5 = (ranks[:, :5] == gt).any(dim=-1).float().mean().item() * 100

    # حفظ أفضل موديل على Google Drive
    save_msg = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), os.path.join(drive_dir, "best_eeg_encoder.pth"))
        save_msg = " -> [تم حفظ أفضل موديل]"

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/40 | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Top-1: {top1:.2f}% | Top-5: {top5:.2f}%{save_msg}")

print("\n=== انتهى التدريب الفعلي بنجاح! ===")

بدء التدريب الفعلي على (cuda) لمدة 40 Epoch...
Epoch 01/40 | Train Loss: 4.8541 | Val Loss: 4.5864 | Top-1: 0.24% | Top-5: 0.48% -> [تم حفظ أفضل موديل]
Epoch 05/40 | Train Loss: 4.3039 | Val Loss: 4.3001 | Top-1: 0.48% | Top-5: 1.62% -> [تم حفظ أفضل موديل]
Epoch 10/40 | Train Loss: 3.4664 | Val Loss: 4.3258 | Top-1: 0.48% | Top-5: 1.80%
Epoch 15/40 | Train Loss: 2.1775 | Val Loss: 4.6281 | Top-1: 0.48% | Top-5: 1.97%
Epoch 20/40 | Train Loss: 1.0326 | Val Loss: 4.9433 | Top-1: 0.36% | Top-5: 1.20%
Epoch 25/40 | Train Loss: 0.5277 | Val Loss: 5.1353 | Top-1: 0.30% | Top-5: 1.50%
Epoch 30/40 | Train Loss: 0.3409 | Val Loss: 5.2386 | Top-1: 0.24% | Top-5: 1.32%
Epoch 35/40 | Train Loss: 0.2771 | Val Loss: 5.2685 | Top-1: 0.18% | Top-5: 1.14%
Epoch 40/40 | Train Loss: 0.2608 | Val Loss: 5.2773 | Top-1: 0.24% | Top-5: 1.08%

=== انتهى التدريب الفعلي بنجاح! ===


In [6]:
class EEGAugmentation(nn.Module):
    def __init__(self, channel_drop_prob=0.15, noise_std=0.15, max_shift=10):
        super().__init__()
        self.channel_drop_prob = channel_drop_prob
        self.noise_std = noise_std
        self.max_shift = max_shift

    def forward(self, x):
        if not self.training:
            return x
        # 1. إطفاء عشوائي لبعض قنوات الدماغ (Spatial Channel Dropout)
        mask = (torch.rand(x.size(0), x.size(1), 1, device=x.device) > self.channel_drop_prob).float()
        x = x * mask / (1.0 - self.channel_drop_prob)

        # 2. إضافة ضوضاء خفيفة لمحاكاة تقلب الإشارة العصبية
        x = x + self.noise_std * torch.randn_like(x)

        # 3. إزاحة زمنية عشوائية بسيطة
        shift = torch.randint(-self.max_shift, self.max_shift + 1, (1,)).item()
        if shift != 0:
            x = torch.roll(x, shifts=shift, dims=-1)
        return x

class EEGEncoder(nn.Module):
    def __init__(self, in_channels=64, embed_dim=768):
        super().__init__()
        self.augment = EEGAugmentation(channel_drop_prob=0.15, noise_std=0.15, max_shift=10)

        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels, 128, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),

            nn.Conv1d(128, 256, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.4),

            nn.Conv1d(256, 512, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(512),
            nn.GELU(),

            nn.AdaptiveAvgPool1d(4),
            nn.Flatten()
        )
        self.projector = nn.Sequential(
            nn.Linear(512 * 4, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(1024, embed_dim)
        )
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

    def forward(self, x):
        x = self.augment(x)
        features = self.encoder(x)
        embeds = self.projector(features)
        return F.normalize(embeds, dim=-1)

print("تم تعريف المعمارية المحسنة (EEGEncoder V2) بنجاح!")

تم تعريف المعمارية المحسنة (EEGEncoder V2) بنجاح!


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = EEGEncoder().to(device)

# تخفيض سرعة التعلم وزيادة الـ Weight Decay لمنع الحفظ العشوائي
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=5e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

def clip_contrastive_loss_v2(eeg_embeds, img_embeds, logit_scale):
    scale = torch.clamp(logit_scale.exp(), max=50.0)
    logits = scale * (eeg_embeds @ img_embeds.T)
    targets = torch.arange(len(eeg_embeds), device=eeg_embeds.device)
    # إضافة Label Smoothing لتعزيز التعميم
    loss_e2i = F.cross_entropy(logits, targets, label_smoothing=0.1)
    loss_i2e = F.cross_entropy(logits.T, targets, label_smoothing=0.1)
    return (loss_e2i + loss_i2e) / 2.0

print(f"بدء التدريب المحسّن (V2) على ({device}) لمدة 50 Epoch...")
best_val_loss = float('inf')
best_top5 = 0.0

for epoch in range(1, 51):
    # --- مرحلة التدريب ---
    model.train()
    train_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        pred_embeds = model(batch_x)
        loss = clip_contrastive_loss_v2(pred_embeds, batch_y, model.logit_scale)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()
    train_loss /= len(train_loader)

    # --- مرحلة التحقق ---
    model.eval()
    val_loss = 0.0
    all_pred, all_true = [], []
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            pred_embeds = model(batch_x)
            loss = clip_contrastive_loss_v2(pred_embeds, batch_y, model.logit_scale)
            val_loss += loss.item()
            all_pred.append(pred_embeds)
            all_true.append(batch_y)
    val_loss /= len(val_loader)

    # حساب الدقة على مجموعة التحقق
    all_pred = torch.cat(all_pred, dim=0)
    all_true = torch.cat(all_true, dim=0)
    sim_matrix = all_pred @ all_true.T
    ranks = sim_matrix.argsort(dim=-1, descending=True)
    gt = torch.arange(len(all_pred), device=device).unsqueeze(1)
    top1 = (ranks[:, :1] == gt).any(dim=-1).float().mean().item() * 100
    top5 = (ranks[:, :5] == gt).any(dim=-1).float().mean().item() * 100

    # حفظ أفضل نسخة على Google Drive
    save_msg = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_top5 = max(best_top5, top5)
        torch.save(model.state_dict(), os.path.join(drive_dir, "best_eeg_encoder_v2.pth"))
        save_msg = " -> [تم حفظ أفضل موديل V2]"

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/50 | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Top-1: {top1:.2f}% | Top-5: {top5:.2f}%{save_msg}")

print(f"\n=== انتهى التدريب المحسّن! أقل Val Loss: {best_val_loss:.4f} | أعلى Top-5: {best_top5:.2f}% ===")

بدء التدريب المحسّن (V2) على (cuda) لمدة 50 Epoch...
Epoch 01/50 | Train Loss: 4.8869 | Val Loss: 4.6461 | Top-1: 0.06% | Top-5: 0.66% -> [تم حفظ أفضل موديل V2]
Epoch 05/50 | Train Loss: 4.7440 | Val Loss: 4.5489 | Top-1: 0.00% | Top-5: 0.42% -> [تم حفظ أفضل موديل V2]
Epoch 10/50 | Train Loss: 4.5333 | Val Loss: 4.4691 | Top-1: 0.36% | Top-5: 1.38% -> [تم حفظ أفضل موديل V2]
Epoch 15/50 | Train Loss: 4.3585 | Val Loss: 4.4663 | Top-1: 0.48% | Top-5: 1.38%
Epoch 20/50 | Train Loss: 4.1976 | Val Loss: 4.4712 | Top-1: 0.36% | Top-5: 1.38%
Epoch 25/50 | Train Loss: 4.0477 | Val Loss: 4.4802 | Top-1: 0.48% | Top-5: 1.62%
Epoch 30/50 | Train Loss: 3.9154 | Val Loss: 4.5074 | Top-1: 0.30% | Top-5: 1.80%
Epoch 35/50 | Train Loss: 3.8232 | Val Loss: 4.5183 | Top-1: 0.42% | Top-5: 1.32%
Epoch 40/50 | Train Loss: 3.7520 | Val Loss: 4.5254 | Top-1: 0.48% | Top-5: 1.62%
Epoch 45/50 | Train Loss: 3.7079 | Val Loss: 4.5313 | Top-1: 0.48% | Top-5: 1.62%
Epoch 50/50 | Train Loss: 3.7109 | Val Loss: 4.53